In [ ]:
import os
import requests
import json
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Create Directory
base_dir = "/content/drive/MyDrive/friends_project_v2"
os.makedirs(base_dir, exist_ok=True)

# 3. Download & Process Data
characters = {
    "monica":   ["Monica Geller", "Monica"],
    "rachel":   ["Rachel Green", "Rachel"],
    "ross":     ["Ross Geller", "Ross"],
    "chandler": ["Chandler Bing", "Chandler"],
    "joey":     ["Joey Tribbiani", "Joey"]
}

print("🚀 Downloading Data...")
base_url = "https://raw.githubusercontent.com/emorynlp/character-mining/master/json/"
seasons = [f"friends_season_{i:02d}.json" for i in range(1, 11)]
data_buffers = {key: [] for key in characters}

for season_file in seasons:
    try:
        response = requests.get(base_url + season_file)
        if response.status_code == 200:
            data = response.json()
            for episode in data['episodes']:
                for scene in episode['scenes']:
                    utterances = scene['utterances']
                    for i in range(1, len(utterances)):
                        curr, prev = utterances[i], utterances[i-1]
                        if not curr['speakers'] or not prev['speakers']: continue
                        speaker, prompt_speaker = curr['speakers'][0], prev['speakers'][0]

                        for char_key, aliases in characters.items():
                            if speaker in aliases and prompt_speaker not in aliases:
                                line = f"User: {prev['transcript']}\n{char_key.capitalize()}: {curr['transcript']}\n<|endoftext|>\n"
                                data_buffers[char_key].append(line)
    except: pass

# Save files locally in Colab for speed
for char, lines in data_buffers.items():
    with open(f"{char}.txt", "w") as f: f.writelines(lines)
    print(f"✅ {char.upper()}: {len(lines)} lines ready.")

Mounted at /content/drive
🚀 Downloading Data...
✅ MONICA: 7621 lines ready.
✅ RACHEL: 8466 lines ready.
✅ ROSS: 8210 lines ready.
✅ CHANDLER: 7691 lines ready.
✅ JOEY: 7363 lines ready.


In [ ]:
!pip install transformers accelerate -q
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

models_dir = "/content/models_output" # Saving to local Colab disk first (Faster than Drive)

for char in characters.keys():
    print(f"\n🎬 Training {char.upper()}...")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2").to("cuda")

    dataset = TextDataset(tokenizer=tokenizer, file_path=f"{char}.txt", block_size=128)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=f"{models_dir}/{char}",
        overwrite_output_dir=True,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        save_steps=1000,
        learning_rate=5e-5,
        report_to="none"
    )

    trainer = Trainer(model=model, args=args, data_collator=data_collator, train_dataset=dataset)
    trainer.train()

    # Save Model
    model.save_pretrained(f"{models_dir}/{char}")
    tokenizer.save_pretrained(f"{models_dir}/{char}")
    print(f"✅ {char} finished!")


🎬 Training MONICA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Step,Training Loss
500,2.607200
1000,2.397100
1500,2.304000


✅ chandler finished!

🎬 Training JOEY...


/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


Step,Training Loss
500,2.640600
1000,2.416700
1500,2.308600


✅ joey finished!


In [ ]:
%%bash

# 1. Install pigz (Parallel GZip) if not already installed
sudo apt-get install pigz -y > /dev/null

# 2. Define Directories
SOURCE_DIR="/content/models_output"
DEST_DIR="/content/drive/MyDrive/friends_models_fast_zips"

# Create destination folder in Drive
mkdir -p "$DEST_DIR"

echo "🚀 Starting High-Speed Individual Zipping..."
echo "📂 Source: $SOURCE_DIR"
echo "💾 Destination: $DEST_DIR"
echo "------------------------------------------------"

# 3. Loop through each character folder and zip it
cd "$SOURCE_DIR"

for dir in */; do
    # Remove trailing slash (e.g., "monica/" -> "monica")
    dirname=${dir%/}

    if [ -d "$dirname" ]; then
        echo "⚡ Zipping $dirname..."

        # The Magic Command: tar + pigz (Uses all CPU cores)
        tar -cf - "$dirname" | pigz -0 -p 4 > "$DEST_DIR/${dirname}.tar.gz"

        echo "   ✅ Saved: ${dirname}.tar.gz"
    fi
done

echo "------------------------------------------------"
echo "🎉 ALL DONE! Check your Google Drive folder: 'friends_models_fast_zips'"


🚀 Starting High-Speed Individual Zipping...
📂 Source: /content/models_output
💾 Destination: /content/drive/MyDrive/friends_models_fast_zips
------------------------------------------------
⚡ Zipping chandler...
   ✅ Saved: chandler.tar.gz
⚡ Zipping joey...
   ✅ Saved: joey.tar.gz
⚡ Zipping monica...
   ✅ Saved: monica.tar.gz
⚡ Zipping rachel...
   ✅ Saved: rachel.tar.gz
⚡ Zipping ross...
   ✅ Saved: ross.tar.gz
------------------------------------------------
🎉 ALL DONE! Check your Google Drive folder: 'friends_models_fast_zips'
